# Notebook B: Combining /accounts, /groups, and /users

**Purpose:** Pull records from three related FOLIO endpoints and join them into a
single analysis-ready DataFrame. This is the kind of cross-record combination the
Stripes UI doesn't do natively.

**Assumed relationships** (confirm against your instance — I have not verified these
field names against current Sunflower schema, so treat them as a starting hypothesis
to check, not a fact):
- `/accounts` records reference a user via a `userId` field.
- `/users` records reference a patron group via a `patronGroup` field (a group's `id`).
- `/groups` records have an `id` and a human-readable `group` name.

If any of those field names are wrong for your instance, the fix is just changing the
column name in the merge step below — the overall approach stays the same.

**How to use this notebook:** Each section has a markdown cell explaining the step,
followed by a code cell. `# TODO (FOLIO-specific)` marks spots to confirm/adjust
against your actual schema.


## 1. Environment setup


In [23]:
# !pip install pandas requests

import pandas as pd
import requests

pd.set_option('display.max_columns', None)


## 2. Login
### Note: This script references patron data, so the login credentials must be able to access user accounts in FOLIO. 


In [24]:

%run folio_auth.ipynb


Login succeeded. Token retrieved.


## 3. A small helper for paginated GET requests

FOLIO endpoints typically page results via `limit`/`offset` query params and return a
JSON object with a records array plus a total count. This helper loops until it has
everything. **Confirm the pagination param names and the response envelope key names
against your instance** — I'm using common FOLIO conventions here, but I have not
verified them against current docs.


In [25]:
def fetch_all_records(endpoint, records_key, limit=100):
    """
    Fetch all records from a paginated FOLIO endpoint.

    endpoint: path like "/accounts", "/groups", "/users"
    records_key: the JSON key holding the list of records, e.g. "accounts", "usergroups", "users"
    """
    all_records = []
    offset = 0
    base_url = "https://api-lse-demo1.folio.ebsco.com"
    headers ={
        "X-Okapi-Tenant": TENANT,
        "Content-Type": "application/json",
    }


    while True:
        response = requests.get(
            f"{base_url}{endpoint}",
            headers=headers,
            params={"limit": limit, "offset": offset},
        )
        response.raise_for_status()  # fail loudly and clearly if something's wrong
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break  # last page
        offset += limit

    return all_records


## 4. Pull data from each endpoint

TODO (FOLIO-specific): confirm the `records_key` for each — FOLIO's convention is
usually the plural of the resource, but it varies (e.g. `/groups` often returns
`"usergroups"` rather than `"groups"` — **check this**, I'm not certain of the exact
key for your instance).


In [26]:
# redefine authenticated fetch that uses existing session, OKAPI_URL, HEADERS, and token
def fetch_all_records(endpoint, records_key, limit=100):
    all_records = []
    offset = 0
    base_url = OKAPI_URL
    headers = HEADERS.copy() if 'HEADERS' in globals() else {"X-Okapi-Tenant": TENANT, "Content-Type": "application/json"}
    if 'token' in globals():
        headers["Authorization"] = f"Bearer {token}"

    while True:
        response = session.get(f"{base_url}{endpoint}", headers=headers, params={"limit": limit, "offset": offset})
        response.raise_for_status()
        payload = response.json()

        batch = payload.get(records_key, [])
        all_records.extend(batch)

        if len(batch) < limit:
            break
        offset += limit

    return all_records

accounts_raw = fetch_all_records("/accounts", records_key="accounts")
groups_raw   = fetch_all_records("/groups",   records_key="usergroups")  
users_raw    = fetch_all_records("/users",    records_key="users")

print(f"accounts: {len(accounts_raw)}")
print(f"groups:   {len(groups_raw)}")
print(f"users:    {len(users_raw)}")
groups_raw   = fetch_all_records("/groups",   records_key="usergroups")  
users_raw    = fetch_all_records("/users",    records_key="users")

print(f"accounts: {len(accounts_raw)}")
print(f"groups:   {len(groups_raw)}")
print(f"users:    {len(users_raw)}")


accounts: 54
groups:   19
users:    133651
accounts: 54
groups:   19
users:    133651


In [27]:
accounts_df = pd.DataFrame(accounts_raw)
groups_df   = pd.DataFrame(groups_raw)
users_df    = pd.DataFrame(users_raw)

accounts_df.head()


,amount,remaining,status,paymentStatus,feeFineType,feeFineOwner,title,callNumber,barcode,materialType,location,metadata,userId,itemId,materialTypeId,feeFineId,ownerId,id,holdingsRecordId,instanceId,contributors,dueDate,loanId,loanPolicyId,overdueFinePolicyId,lostItemFeePolicyId,returnedDate
0,27.5,0.0,{'name': 'Closed'},{'name': 'Transferred fully'},Damaged item,Billing Office,The Economic Sociology of Development / Andrew...,,111222333,Book,Main Stacks,{'createdDate': '2024-12-05T21:27:43.484+00:00...,cd58d70f-78ac-4438-ae3c-6016c9147f1e,73db06cb-36e2-48db-8db9-0cc1ea835b5f,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,66c16fdc-809f-4d5d-b1b9-3611480d7d86,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,955e1fad-9899-4a87-81c4-2f97938cdd56,26df6234-fd57-4d3e-9c5e-c765aa58a570,f5c2e43b-8340-5d93-8c8b-63510a7d2950,"[{'name': 'Schrank, Andrew'}]",NaN,NaN,NaN,NaN,NaN,NaN
1,10.0,0.0,{'name': 'Closed'},{'name': 'Cancelled item returned'},Lost item processing fee,Billing Office,25 Common Core math lessons for the interactiv...,NaN,32260010586222,Book,Interlibrary Loan,{'createdDate': '2025-06-22T00:47:09.642+00:00...,9e8f2429-e1ef-4de1-aa16-02dce79190f8,1f90e5ca-f971-43fe-a7b1-7ff5cced34ac,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,c7dede15-aa48-45ed-860b-f996540180e0,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,996c7869-519a-4112-aca0-bf13f3277fd0,6758542a-0d75-47e0-bb77-dc0d694a1c26,251af7cd-9381-4a3c-b751-0f6dfec0776c,[],2025-03-31T23:59:00.000+00:00,616ddee0-9ea2-43c8-98be-32eded8a976e,1a73c74f-3736-4bb8-95ce-b5e9555ddfb4,cf4a7211-809a-4512-b98c-c2515793c526,84143c48-b6ee-4c45-ba4a-a56545816b26,NaN
2,25.0,0.0,{'name': 'Closed'},{'name': 'Cancelled item returned'},Lost item processing fee,Billing Office,Sustainability in industry 4.0 : challenges an...,NaN,111222332,Book,Main Stacks,{'createdDate': '2024-12-09T20:17:55.558+00:00...,c4b64a1a-db66-4a7a-8ac6-427ab077308e,283b1a7a-8274-4fcd-b743-6f569ad59e10,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,c7dede15-aa48-45ed-860b-f996540180e0,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,1b867736-8868-4946-b052-02297f7ae22b,70a2052d-2826-4be0-ba3c-a5806948040d,91ccccf7-c43b-54bb-8c12-3ded1e5fc204,"[{'name': 'Avikal, Shwetank'}, {'name': 'Singh...",2024-12-09T20:08:08.825+00:00,26f88554-97af-443e-ae77-576ace3dd597,a4bb6eed-1a88-4931-bb4b-2858f985c3f5,cf4a7211-809a-4512-b98c-c2515793c526,1e1a6a28-6565-4747-a655-14a2fa0aaae5,NaN
3,10.0,0.0,{'name': 'Closed'},{'name': 'Transferred fully'},Lost item processing fee,Billing Office,"Degas : the complete etchings, lithographs and...",MT580,32260008852180,Book,Main Stacks,{'createdDate': '2025-04-20T00:55:14.924+00:00...,9e8f2429-e1ef-4de1-aa16-02dce79190f8,e761cef6-4e4b-400c-9736-e521ebe0690f,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,c7dede15-aa48-45ed-860b-f996540180e0,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,faa7c68a-9c4c-47a6-9e0b-2b9bd4484a15,b4fd5dd7-2be5-4396-8324-fe52daa7d7b1,5375ce98-c179-49a0-a6d3-dff248edf458,"[{'name': 'Degas, Edgar, 1834-1917'}, {'name':...",2025-01-29T23:59:59.000+00:00,fe9b5453-d7bd-44d8-9b53-a48e9ca69a05,1a73c74f-3736-4bb8-95ce-b5e9555ddfb4,30d1554f-51c2-491c-b419-901e22361a0b,84143c48-b6ee-4c45-ba4a-a56545816b26,NaN
4,10.0,0.0,{'name': 'Closed'},{'name': 'Cancelled item returned'},Lost item processing fee,Billing Office,Guitar exercises / by Mark Phillips and Jon Ch...,MT588,092384,Book,Main Stacks,{'createdDate': '2025-03-08T00:27:03.998+00:00...,1932662b-e355-4c6a-bdd7-816b84f303de,4994e1dc-b29a-40a3-a918-da4b3c62d141,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,c7dede15-aa48-45ed-860b-f996540180e0,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,50133971-c3c2-49bb-be67-f8fa98134f3b,1e2402e6-b9da-476d-88c2-492892818c1a,b0312e52-b460-41ac-9a6a-f74d87a5c339,"[{'name': 'Phillips, Mark, 1947-'}, {'name': '...",2025-01-17T23:59:59.000+00:00,406171f6-ae09-48bc-8f99-096dd8b5b8b7,1a73c74f-3736-4bb8-95ce-b5e9555ddfb4,30d1554f-51c2-491c-b419-901e22361a0b,84143c48-b6ee-4c45-ba4a-a56545816b26,NaN


## 5. Inspect join keys before merging

This is the step worth slowing down for. Before merging, check:
- Do the key columns actually exist under the names you expect?
- Are the data types consistent between the two sides of each join (e.g. both strings)?
- Any leading/trailing whitespace or case differences?


In [28]:
print(accounts_df.columns.tolist())
print(users_df.columns.tolist())
print(groups_df.columns.tolist())


['amount', 'remaining', 'status', 'paymentStatus', 'feeFineType', 'feeFineOwner', 'title', 'callNumber', 'barcode', 'materialType', 'location', 'metadata', 'userId', 'itemId', 'materialTypeId', 'feeFineId', 'ownerId', 'id', 'holdingsRecordId', 'instanceId', 'contributors', 'dueDate', 'loanId', 'loanPolicyId', 'overdueFinePolicyId', 'lostItemFeePolicyId', 'returnedDate']
['username', 'id', 'active', 'type', 'departments', 'proxyFor', 'personal', 'createdDate', 'updatedDate', 'metadata', 'preferredEmailCommunication', 'externalSystemId', 'barcode', 'patronGroup', 'expirationDate', 'customFields', 'enrollmentDate', 'tags']
['group', 'desc', 'id', 'metadata', 'expirationOffsetInDays']


In [29]:
# Spot-check types of the columns you intend to join on
# TODO (FOLIO-specific): adjust column names if yours differ
print(accounts_df['userId'].dtype)
print(users_df['id'].dtype)
print(users_df['patronGroup'].dtype)
print(groups_df['id'].dtype)


str
str
str
str


## 6. Merge

Two joins: accounts → users, then that result → groups.

Starting with `how='left'` keeps every account row even if a match isn't found, so you
can see what didn't match rather than silently losing rows.


In [30]:
# Step 1: accounts + users
accounts_users = accounts_df.merge(
    users_df,
    left_on='userId',
    right_on='id',
    how='left',
    suffixes=('_account', '_user'),
)

# Step 2: + groups
full_df = accounts_users.merge(
    groups_df,
    left_on='patronGroup',
    right_on='id',
    how='left',
    suffixes=('', '_group'),
)

full_df.head()


,amount,remaining,status,paymentStatus,feeFineType,feeFineOwner,title,callNumber,barcode_account,materialType,location,metadata_account,userId,itemId,materialTypeId,feeFineId,ownerId,id_account,holdingsRecordId,instanceId,contributors,dueDate,loanId,loanPolicyId,overdueFinePolicyId,lostItemFeePolicyId,returnedDate,username,id_user,active,type,departments,proxyFor,personal,createdDate,updatedDate,metadata_user,preferredEmailCommunication,externalSystemId,barcode_user,patronGroup,expirationDate,customFields,enrollmentDate,tags,group,desc,id,metadata,expirationOffsetInDays
0,27.5,0.0,{'name': 'Closed'},{'name': 'Transferred fully'},Damaged item,Billing Office,The Economic Sociology of Development / Andrew...,,111222333,Book,Main Stacks,{'createdDate': '2024-12-05T21:27:43.484+00:00...,cd58d70f-78ac-4438-ae3c-6016c9147f1e,73db06cb-36e2-48db-8db9-0cc1ea835b5f,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,66c16fdc-809f-4d5d-b1b9-3611480d7d86,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,955e1fad-9899-4a87-81c4-2f97938cdd56,26df6234-fd57-4d3e-9c5e-c765aa58a570,f5c2e43b-8340-5d93-8c8b-63510a7d2950,"[{'name': 'Schrank, Andrew'}]",NaN,NaN,NaN,NaN,NaN,NaN,654654654,cd58d70f-78ac-4438-ae3c-6016c9147f1e,False,patron,[],[],"{'lastName': 'Whitehair', 'firstName': 'Kristi...",2025-05-12T14:31:38.451+00:00,2025-06-12T13:09:20.569+00:00,{'createdDate': '2024-11-12T21:33:43.471+00:00...,[],NaN,654654654,04f8f36c-9ef5-41ab-8bfb-841e509f8b52,2025-12-24T23:59:59.000+00:00,{},2025-05-02T00:00:00.000+00:00,NaN,Undergrad,Undergraduate students,04f8f36c-9ef5-41ab-8bfb-841e509f8b52,{'createdDate': '2024-06-25T16:09:25.098+00:00...,30.0
1,10.0,0.0,{'name': 'Closed'},{'name': 'Cancelled item returned'},Lost item processing fee,Billing Office,25 Common Core math lessons for the interactiv...,NaN,32260010586222,Book,Interlibrary Loan,{'createdDate': '2025-06-22T00:47:09.642+00:00...,9e8f2429-e1ef-4de1-aa16-02dce79190f8,1f90e5ca-f971-43fe-a7b1-7ff5cced34ac,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,c7dede15-aa48-45ed-860b-f996540180e0,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,996c7869-519a-4112-aca0-bf13f3277fd0,6758542a-0d75-47e0-bb77-dc0d694a1c26,251af7cd-9381-4a3c-b751-0f6dfec0776c,[],2025-03-31T23:59:00.000+00:00,616ddee0-9ea2-43c8-98be-32eded8a976e,1a73c74f-3736-4bb8-95ce-b5e9555ddfb4,cf4a7211-809a-4512-b98c-c2515793c526,84143c48-b6ee-4c45-ba4a-a56545816b26,NaN,TestMike,9e8f2429-e1ef-4de1-aa16-02dce79190f8,True,staff,[],[],"{'lastName': 'Abrahamson', 'firstName': 'Mike'...",2026-06-18T15:54:42.656+00:00,2026-06-18T15:54:42.656+00:00,{'createdDate': '2024-04-24T16:15:42.357+00:00...,[],123412341234,123412341234,b467d748-5964-431b-907c-5f320703e92d,2030-11-01T23:59:59.999+00:00,{},NaN,{'tagList': []},zEBSCO Support Group,Group for EBSCO users for tenant fs00001208,b467d748-5964-431b-907c-5f320703e92d,{'createdDate': '2024-04-10T06:51:34.217+00:00...,NaN
2,25.0,0.0,{'name': 'Closed'},{'name': 'Cancelled item returned'},Lost item processing fee,Billing Office,Sustainability in industry 4.0 : challenges an...,NaN,111222332,Book,Main Stacks,{'createdDate': '2024-12-09T20:17:55.558+00:00...,c4b64a1a-db66-4a7a-8ac6-427ab077308e,283b1a7a-8274-4fcd-b743-6f569ad59e10,f1f8036d-b573-4c73-85fc-83c1ddf4dbe0,c7dede15-aa48-45ed-860b-f996540180e0,59717958-8e4c-4f93-a4b3-5ab81b9cb1f0,1b867736-8868-4946-b052-02297f7ae22b,70a2052d-2826-4be0-ba3c-a5806948040d,91ccccf7-c43b-54bb-8c12-3ded1e5fc204,"[{'name': 'Avikal, Shwetank'}, {'name': 'Singh...",2024-12-09T20:08:08.825+00:00,26f88554-97af-443e-ae77-576ace3dd597,a4bb6eed-1a88-4931-bb4b-2858f985c3f5,cf4a7211-809a-4512-b98c-c2515793c526,1e1a6a28-6565-4747-a655-14a2fa0aaae5,NaN,NaN,c4b64a1a-db66-4a7a-8ac6-427ab077308e,False,patron,[],[],"{'lastName': 'McBlockerson', 'firstName': 'Blo...",2024-12-23T14:45:01.913+00:00,2024-12-23T14:45:01.913+00:00,{'createdDate': '2024-12-09T19:17:56.668+00:00...,[],NaN,666777444,c0f2ea7f-7510-47f3-9116-c703b92a4e89,2025-12-09T23:59:59.000+00:00,{},NaN,NaN,Blocks-PatronGroup,use to test automated patron blocks,c0f2ea7f-7510

## 7. Validate the merge

Common beginner pitfall: a one-to-many relationship silently multiplying rows, or a
type mismatch causing everything to come back unmatched. Check both.


In [31]:
print("Original accounts rows:", len(accounts_df))
print("After merging with users:  ", len(accounts_users))
print("After merging with groups: ", len(full_df))

# Rows where the user or group match failed — worth investigating, not ignoring
unmatched_users = full_df[full_df['id_user'].isnull()] if 'id_user' in full_df.columns else pd.DataFrame()
print("Accounts with no matching user:", len(unmatched_users))


Original accounts rows: 54
After merging with users:   54
After merging with groups:  54
Accounts with no matching user: 5


## 8. Analyze the combined dataset

Now that accounts, users, and groups are joined, you can ask questions that span all
three — e.g. total fee/fine amounts by patron group. Adjust field names to match your
actual `/accounts` schema (commonly something like `amount` or `remaining`).


In [ ]:
# TODO (FOLIO-specific): confirm the actual fee/fine amount field name
summary = full_df.groupby('group')['remaining'].sum().sort_values(ascending=False)
print(summary)
